# Zhang-style GNNHAR-IV Scale Experiment

This launcher runs the corrected rolling-window S&P 100 / S&P 500 scale experiment. It is intended to replace the earlier static-graph scale screen when answering whether larger asset universes increase graph-structure gains.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, pathlib, subprocess, json, textwrap, shutil, time

REPO = pathlib.Path('/content/GNNHAR')
BRANCH = '2026-06-01'
RUN_ID = time.strftime('scale-zhang-roll-%Y%m%dT%H%M%SZ', time.gmtime())
OUT_ROOT = pathlib.Path('/content/GNNHAR-colab-runs') / RUN_ID
OUT_ROOT.mkdir(parents=True, exist_ok=True)
print('RUN_ID =', RUN_ID)
print('OUT_ROOT =', OUT_ROOT)


In [ ]:
if not REPO.exists():
    subprocess.run([
        'git', 'clone', '--branch', BRANCH,
        'https://github.com/easygl1der/GNNHAR.git', str(REPO)
    ], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', BRANCH], check=True)

os.chdir(REPO)
subprocess.run(['git', 'log', '--oneline', '-5'], check=True)
subprocess.run(['pip', 'install', '-q', '-r', 'requirements-scale.txt'], check=True)


## Data and Method Audit

This confirms whether RV, IV, and return panels are paired and whether the previous S&P500 run used a real GLASSO graph.


In [ ]:
subprocess.run([
    'python', 'scripts/analysis/audit_scale_experiment_data.py',
    '--output-dir', str(OUT_ROOT / 'audit')
], check=True)
print((OUT_ROOT / 'audit' / 'scale_data_method_audit.md').read_text()[:3000])


## Smoke Test

Run one rolling block on small universes. This checks code paths only; do not interpret the smoke losses as research results.


In [ ]:
smoke_cmds = [
    [
        'python', 'scripts/analysis/gnnhar_iv_zhang_scale_pipeline.py',
        '--universe-name', 'sp100-smoke',
        '--data-dir', 'data/scale_experiment/sp100',
        '--returns-file', 'data/scale_experiment/sp100/daily_returns.csv',
        '--output-dir', str(OUT_ROOT / 'smoke' / 'sp100'),
        '--coverage-threshold', '0.95',
        '--max-tickers', '20',
        '--max-blocks', '1',
        '--models', 'HAR,GHAR,GHAR2H,GHAR3H,HAR+IV,GHAR+IV,GHAR2H+IV,GHAR3H+IV,GNNHAR1L,GNNHAR1L-IV',
        '--hidden-grid', '9',
        '--lr-grid', '0.001',
        '--epochs', '40',
        '--num-nn', '1',
        '--mcs-bootstrap', '20',
        '--skip-figures',
        '--fast'
    ],
    [
        'python', 'scripts/analysis/gnnhar_iv_zhang_scale_pipeline.py',
        '--universe-name', 'sp500-smoke',
        '--data-dir', 'data/scale_experiment/sp500',
        '--returns-file', 'data/scale_experiment/sp500/daily_returns.csv',
        '--output-dir', str(OUT_ROOT / 'smoke' / 'sp500'),
        '--coverage-threshold', '0.99',
        '--max-tickers', '25',
        '--max-blocks', '1',
        '--models', 'HAR,GHAR,GHAR2H,GHAR3H,HAR+IV,GHAR+IV,GHAR2H+IV,GHAR3H+IV,GNNHAR1L',
        '--hidden-grid', '9',
        '--lr-grid', '0.001',
        '--epochs', '30',
        '--num-nn', '1',
        '--mcs-bootstrap', '20',
        '--skip-figures',
        '--fast'
    ]
]
for cmd in smoke_cmds:
    subprocess.run(cmd, check=True)


## Full S&P 100 Run

This is the first interpretable corrected run. It uses rolling GLASSO and Zhang-style GNNHAR architecture.


In [ ]:
subprocess.run([
    'python', 'scripts/analysis/gnnhar_iv_zhang_scale_pipeline.py',
    '--universe-name', 'sp100',
    '--data-dir', 'data/scale_experiment/sp100',
    '--returns-file', 'data/scale_experiment/sp100/daily_returns.csv',
    '--output-dir', str(OUT_ROOT / 'full' / 'sp100'),
    '--coverage-threshold', '0.95',
    '--models', 'HAR,GHAR,GHAR2H,GHAR3H,HAR+IV,GHAR+IV,GHAR2H+IV,GHAR3H+IV,GNNHAR1L,GNNHAR2L,GNNHAR3L,GNNHAR1L-IV,GNNHAR2L-IV,GNNHAR3L-IV',
    '--hidden-grid', '9,16,32',
    '--lr-grid', '0.001,0.0003',
    '--epochs', '300',
    '--num-nn', '1',
    '--mcs-bootstrap', '80'
], check=True)


## Full S&P 500 Run

This pass runs the same GHAR, multi-hop GHAR, GNNHAR1L/2L/3L, and IV-augmented specifications on the S&P 500 panel. It keeps the coverage-filtered universe instead of capping nodes, so it is the main node-scale test.


In [ ]:
subprocess.run([
    'python', 'scripts/analysis/gnnhar_iv_zhang_scale_pipeline.py',
    '--universe-name', 'sp500',
    '--data-dir', 'data/scale_experiment/sp500',
    '--returns-file', 'data/scale_experiment/sp500/daily_returns.csv',
    '--output-dir', str(OUT_ROOT / 'full' / 'sp500'),
    '--coverage-threshold', '0.99',
    '--models', 'HAR,GHAR,GHAR2H,GHAR3H,HAR+IV,GHAR+IV,GHAR2H+IV,GHAR3H+IV,GNNHAR1L,GNNHAR2L,GNNHAR3L,GNNHAR1L-IV,GNNHAR2L-IV,GNNHAR3L-IV',
    '--hidden-grid', '9,16',
    '--lr-grid', '0.001,0.0003',
    '--epochs', '220',
    '--num-nn', '1',
    '--mcs-bootstrap', '60'
], check=True)


## Rolling Scale Summary

Run this after the full S&P100 and S&P500 cells finish.


In [ ]:
summary_dir = OUT_ROOT / 'summary'
run_dirs = [OUT_ROOT / 'full' / 'sp100', OUT_ROOT / 'full' / 'sp500']
run_dirs = [p for p in run_dirs if (p / 'tables' / 'model_losses.csv').exists()]
if len(run_dirs) >= 1:
    cmd = [
        'python', 'scripts/analysis/summarize_zhang_scale_experiment.py',
        '--run-dirs', *map(str, run_dirs),
        '--labels', *[p.name for p in run_dirs],
        '--output-dir', str(summary_dir)
    ]
    subprocess.run(cmd, check=True)
    import pandas as pd
    display(pd.read_csv(summary_dir / 'rolling_scale_summary.csv'))
    display(pd.read_csv(summary_dir / 'graph_audit.csv'))
else:
    print('No full run directories found yet.')


## Summarize Key Files


In [ ]:
for path in sorted(OUT_ROOT.rglob('run_metadata.json')):
    print('---', path)
    meta = json.loads(path.read_text())
    print(json.dumps({
        'implementation': meta.get('implementation'),
        'panel': meta.get('panel', {}),
        'rolling': meta.get('rolling', {})
    }, indent=2)[:2000])

for path in sorted(OUT_ROOT.rglob('model_losses.csv')):
    print('---', path)
    import pandas as pd
    display(pd.read_csv(path).head(12))

for path in sorted(OUT_ROOT.rglob('graph_blocks.csv')):
    print('---', path)
    display(pd.read_csv(path).head())
